# What Comes Next

You just spent four hours building a pipeline that took a document, chunked it,
embedded it, retrieved from it, evaluated it, and adapted a model against it.
That is real. The pipeline works. The eval scores are real scores.

And it was four hours.

What follows is an honest map of what a production engagement with this
technology actually looks like.

---

## What You Built Is a Mental Model

The lab was not designed to produce a deliverable. It was designed to give you
felt experience with a problem space that your customers are trying to navigate,
so you can ask better questions, spot bad assumptions earlier, and know when to
escalate to the people who do the build.

That is a different kind of value than a working system. It is also the right
kind for your role.

Production is a different problem than the lab. Not a harder version of the same
problem. A different one.

The difference is not the model. It is everything around the model.

---

## This is a Process and Works as a Chain

Every stage of the pipeline depends on the stage before it. Bad ingestion
produces bad chunks. Bad chunks produce bad embeddings. Bad embeddings produce
bad retrieval. Bad retrieval means the model never sees the right information,
and no amount of fine-tuning will fix that.

This is not a collection of independent problems. It is a chain. And a chain is
only as strong as its weakest link.

In the lab, we controlled for this. The document was clean. The format was
consistent. The evaluation questions were chosen to be answerable. Real
engagements do not come pre-cleaned.

The work of a real engagement is largely the work of finding the weak link,
fixing it, and confirming the fix improved the system. Then finding the next
one. That process takes time not because the technology is immature, but because
the problem is genuinely complex and the complexity lives in the details of each
customer's specific corpus, questions, and definition of correct.

---

### Corpus deduplication
In the multi-document section of the RAG evaluation, three retrieval
slots were consumed by three versions of the same rulebook. The current edition
was not retrieved at all. In a customer corpus with hundreds of versioned
documents, this problem gets worse, not better. The fix is upstream corpus
curation, not a model change.

### Metadata filtering
Once you have a multi-document corpus, retrieval needs
to be smarter than "find the most similar chunks." You often need to filter by
document type, date range, department, or authority level before ranking by
similarity. This requires metadata strategy at ingestion time. Without it, a
question about current policy might return a chunk from a three-year-old version
that scored higher on similarity.

### Chunking strategy comparison
The lab used a single chunking approach. In
practice, the right strategy depends on document structure. Fixed-size chunking,
semantic chunking, hierarchical chunking, and document-structure-aware chunking
produce meaningfully different retrieval behavior against different content types.
A chunk boundary that splits a table in half means neither piece is retrievable
on its own, and no retrieval tuning will fix that.

### LLM-as-judge evaluation
Keyword matching catches clear failures. It misses
nuanced correctness issues. Production evaluation typically requires a second
model evaluating the first model's answers against reference answers, with human
spot-checking on a sample.

### Embedding model selection
We used a specific Granite embedding model. In
production, embedding model choice depends on domain vocabulary, context length
requirements, multilingual needs, and latency constraints. This is a
benchmarking exercise, not a default. A model trained on general web text can
consistently fail to retrieve the right chunks simply because the domain uses
technical terminology the embedding space has never seen.

### Reranking
Retrieval returns candidates. Reranking re-scores them using a
separate, more expensive model before passing them to the generator. It
consistently improves answer quality in production systems and is consistently
skipped in early-stage work.

### Input sanitization
Every query that reaches your model is a potential attack surface. Prompt
injection, where a user crafts input that overrides the system prompt or
manipulates the model into ignoring its instructions, is the most discussed
risk, but it is not the only one. Unsanitized input can also trigger unintended
tool execution, leak system prompt content, or cause the model to return data
from outside the authorized corpus.

In a RAG system, the risk is compounded. User input flows into the retrieval
query, which means a malicious query can influence what context the model sees.
A carefully constructed input might retrieve chunks that, when combined with the
injection payload, produce outputs the system was never intended to generate.

The mitigations are straightforward in principle and require discipline in
practice. Validate and sanitize all user input before it reaches the model or
the retrieval layer. Enforce length limits. Strip or escape control characters
and known injection patterns. Separate the system prompt from user content in
ways that the model architecture respects. Log inputs so you can audit what the
system was asked and how it responded.

None of this is optional in a production system. The lab environment is
controlled. Customer environments are not. The question is not whether someone
will try to break the system. It is whether you will notice when they do.

Each of these is a place where the chain can break.

---

## What the Lab Did Not Show You

The lab was designed to teach a workflow. It was not designed to simulate
production. The following topics were intentionally omitted to keep the lab
focused, but they are not optional in a real deployment. Each one represents a
category of failure that does not exist in a lab setting and will absolutely
exist in the field.

### Output guardrails
Input sanitization handles what goes into the model. Output guardrails handle
what comes out. A model connected to a customer corpus can leak personally
identifiable information, generate hallucinated citations that look real,
reference documents the user is not authorized to see, or produce answers that
are technically grounded but contextually inappropriate.

The fix is a validation layer between the model and the user. Check outputs for
PII patterns before returning them. Verify that cited sources actually exist in
the corpus and that the user has access to them. Flag answers where the model's
confidence is low or where the retrieved context does not support the claim.
Consider a second, smaller model whose only job is to evaluate whether the
primary model's output is safe to return.

This is not overcaution. It is the difference between a system that works in a
demo and a system that survives contact with compliance, legal, and real users
who will find every edge case you did not anticipate.

### Document freshness and re-ingestion
The lab treated the corpus as a fixed artifact. You ingested it once, embedded
it once, and queried against it for the rest of the session. Production corpora
are not fixed. Documents are updated, superseded, retracted, and versioned.

When a document changes, the old embeddings are still in the vector store. The
old chunks are still retrievable. Unless you have a re-ingestion pipeline that
detects changes, re-processes affected documents, removes stale embeddings, and
validates that retrieval still returns current information, your system will
silently serve outdated answers with full confidence.

The hard part is not the re-ingestion itself. It is knowing that a document
changed. In enterprise environments, documents live in SharePoint, Confluence,
S3 buckets, shared drives, and email attachments. Building a reliable change
detection layer across those sources is often the most underestimated piece of
the production architecture.

### Access control in retrieval
In the lab, every participant could see every document. In production, that is
almost never true. A RAG system that retrieves chunks without respecting
document-level permissions is a data exfiltration tool with a friendly
interface.

The problem is architectural. The vector store does not natively understand who
is allowed to see what. Access control must be enforced either at query time, by
filtering retrieval results against the user's permissions before they reach the
model, or at ingestion time, by maintaining separate indices per access tier.
Both approaches have trade-offs. Query-time filtering is simpler but slower at
scale. Ingestion-time separation is faster but harder to maintain as permissions
change.

The conversation with the customer about this topic should happen early, not
after the first security review. Ask who sees what. Ask how permissions are
managed today. Ask what happens when someone leaves a team. The answers shape
the retrieval architecture more than any model choice will.

### Observability and drift detection
The lab gave you evaluation as a diagnostic tool. You ran it, looked at the
results, and made decisions. In production, you need evaluation that runs
continuously without anyone looking at it, and alerts when something changes.

Model drift is real. Retrieval quality degrades as the corpus grows and the
distribution of content shifts away from what the embedding model was optimized
for. Answer quality changes when the underlying model is updated by the
provider. Latency increases as the vector store grows. None of these failures
announce themselves. They accumulate gradually until a user reports that the
system used to work better, and no one can pinpoint when or why.

The fix is observability infrastructure. Log every query, every retrieval
result, every model response, and every latency measurement. Run your evaluation
set on a schedule and track scores over time. Set thresholds that trigger alerts
when retrieval distances drift upward or answer quality scores drift downward.
Treat your RAG system like any other production service: if you are not
monitoring it, you do not know if it is working.

### Caching and latency
The lab processed one query at a time with no concern for response time. In
production, users expect sub-second responses, and the system may serve hundreds
of concurrent requests.

Semantic caching addresses the most common case: many users ask variations of
the same question. Rather than running the full retrieval and generation pipeline
every time, cache the response for semantically similar queries and return it
directly. This reduces latency, cuts inference costs, and decreases load on the
model endpoint.

The implementation requires a similarity threshold for cache hits, a TTL
strategy that accounts for corpus freshness, and invalidation logic that clears
cached responses when the underlying documents change. Get the threshold wrong
and you serve stale or incorrect cached answers. Get the TTL wrong and you lose
the cost benefit. Like most production optimizations, it is simple in concept
and requires careful tuning in practice.

### User feedback loops
The evaluation set you built in the lab was authored by the people who built the
system. In production, the most valuable signal about answer quality comes from
the people who use the system every day.

A thumbs-up/thumbs-down mechanism on each response is the minimum. Better is a
correction flow where users can flag a wrong answer and provide the right one.
Best is a pipeline that routes those corrections back into the evaluation set,
so the questions real users actually care about become the questions the system
is measured against.

Without this loop, the evaluation set fossilizes. It measures what mattered when
the system launched, not what matters now. The system can score 9/10 on the
original eval while silently failing on the questions users actually ask. User
feedback is what keeps the evaluation honest and the system aligned with the
problem it is supposed to solve.

---

## The Conversation to Have with Your Customer

When a customer says they want to build a RAG system, the instinct is to talk
about models. The right conversation starts somewhere else.

Ask them: how are you measuring correct?

If they cannot answer that, neither can you. And any change you make, to
retrieval, to chunking, to the model, becomes unjustifiable, because there is no
before and after to compare.

The evaluation set you build with their subject matter experts, even ten or
twenty questions with agreed-upon reference answers, is often the most productive
artifact of the first month. It forces the customer to articulate standards they
have never written down. It gives you a shared definition of done. And it gives
every subsequent recommendation a defensible basis.

When they ask whether they need fine-tuning, the answer that earns trust is not
yes or no. It is: "Here is what the evaluation shows. Here is what is failing
and why. Here is what we would try before we consider changing the model."

That positions you as someone who solves problems methodically, not someone who
sells the most expensive option.

---

## Further Reading

### Retrieval and RAG

[RAGAS: Automated Evaluation of Retrieval Augmented Generation](https://arxiv.org/abs/2309.15217)
(paper) -- the academic basis for automated RAG evaluation, introducing metrics
that do not require human-annotated ground truth.

[LangChain Text Splitters documentation](https://python.langchain.com/docs/concepts/text_splitters/)
-- practical comparisons of fixed-size, semantic, and hierarchical chunking
approaches with code examples.

[Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172)
(paper) -- explains why what you retrieve and where it appears in the prompt
both matter. Performance degrades significantly when relevant information appears
in the middle of a long context.

### Evaluation

[Hugging Face TRL documentation](https://huggingface.co/docs/trl/index) -- the
library used in the remaining sections, with production-grade examples for training and
evaluation workflows.

[MLflow documentation](https://mlflow.org/docs/latest/index.html) -- experiment
tracking and model registry, relevant to managing iteration cycles across a real
engagement.

### Fine-Tuning and Model Adaptation

[LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
(paper) -- the foundational paper for the adaptation technique used in the remaining sections.
Explains why you can train a tiny fraction of parameters and still get
meaningful results.

[LIMA: Less Is More for Alignment](https://arxiv.org/abs/2305.11206) (paper) --
evidence that data quality matters more than data quantity. A model fine-tuned
on 1,000 carefully curated examples outperformed models trained on far larger
datasets.

### Production Considerations

[Building LLM Applications for Production](https://huyenchip.com/2023/04/11/llm-engineering.html)
(Chip Huyen) -- one of the most referenced practical guides on the gap between
demos and production. The core observation maps directly to what you just built:
it is easy to make something cool with LLMs, and very hard to make something
production-ready.

---

*This document is part of the extras folder. Nothing here is required.
All of it is real.*